<a href="https://colab.research.google.com/github/kleitonfr/apache-spark-rdd/blob/main/LeituraEscritaDeDados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

import pyspark
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import date

# 1. Primeiros passos com RDD

In [ ]:
sc = SparkContext.getOrCreate()

rdd = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
rdd.getNumPartitions()

2

In [ ]:
rdd.first()

1

In [ ]:
rdd.take(5)

[1, 2, 3, 4, 5]

In [ ]:
rdd.collect() # nesse caso, porque atuo com um conjunto pequeno de dados

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [ ]:
# Trabalhando com outro conjunto de dados
rddTexto = sc.parallelize(["Spark", "Hadoop", "Python", "Spark", "Hadoop"])

rddMinusculo = rddTexto.map(lambda elemento:elemento.lower()) #salva novo RDD, outras patições distintas das originais
rddMinusculo.take(5)

['spark', 'hadoop', 'python', 'spark', 'hadoop']

In [ ]:
sc.stop()

# 2. Transformações e Ações em RDD - Pipeline de Contagem de **Palavras**

In [97]:
conf = SparkConf().setAppName("Leitura de arquivo texto")
sc = SparkContext(conf=conf).getOrCreate()

In [98]:
rdd = sc.textFile("sample_data/README.md")
rdd.getNumPartitions()

2

In [ ]:
rdd.count()

19

In [ ]:
rdd.take(10)

['This directory includes a few sample datasets to get you started.',
 '',
 '*   `california_housing_data*.csv` is California housing data from the 1990 US',
 '    Census; more information is available at:',
 '    https://docs.google.com/document/d/e/2PACX-1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia_DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN/pub',
 '',
 '*   `mnist_*.csv` is a small sample of the',
 '    [MNIST database](https://en.wikipedia.org/wiki/MNIST_database), which is',
 '    described at: http://yann.lecun.com/exdb/mnist/',
 '']

In [99]:
palavra = rdd.map(lambda x: x.split(' '))
palavra.take(5)

[['This',
  'directory',
  'includes',
  'a',
  'few',
  'sample',
  'datasets',
  'to',
  'get',
  'you',
  'started.'],
 [''],
 ['*',
  '',
  '',
  '`california_housing_data*.csv`',
  'is',
  'California',
  'housing',
  'data',
  'from',
  'the',
  '1990',
  'US'],
 ['', '', '', '', 'Census;', 'more', 'information', 'is', 'available', 'at:'],
 ['',
  '',
  '',
  '',
  'https://docs.google.com/document/d/e/2PACX-1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia_DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN/pub']]

In [100]:
palavra = rdd.flatMap(lambda x: x.split(' '))
palavra.take(5) # quebra em várias listas e depois achata

['This', 'directory', 'includes', 'a', 'few']

In [101]:
palavraMinuscula = palavra.map(lambda x: x.lower())
print('Map: ', palavraMinuscula.take(20))

Map:  ['this', 'directory', 'includes', 'a', 'few', 'sample', 'datasets', 'to', 'get', 'you', 'started.', '', '*', '', '', '`california_housing_data*.csv`', 'is', 'california', 'housing', 'data']


In [102]:
palavraMinuscula = palavra.flatMap(lambda x: x.lower())
print('Map: ', palavraMinuscula.take(20))

Map:  ['t', 'h', 'i', 's', 'd', 'i', 'r', 'e', 'c', 't', 'o', 'r', 'y', 'i', 'n', 'c', 'l', 'u', 'd', 'e']


In [103]:
palavrasComecaT = palavraMinuscula.filter(lambda x: x.startswith('t'))
print(palavrasComecaT.take(5))

['t', 't', 't', 't', 't']


In [104]:
# preparando chave-valor para contar:

palavraChaveValor = palavra.map(lambda x: (x, 1))
palavraChaveValor.take(5)

[('This', 1), ('directory', 1), ('includes', 1), ('a', 1), ('few', 1)]

In [105]:
# somando por chave com reduceByKey

palavraContar = palavraChaveValor.reduceByKey(lambda x, y: x + y)
palavraContarOrd = palavraContar.sortByKey(ascending=-1)
palavraContarOrd.take(20)

[('', 51),
 ("'Graphs", 1),
 ('(1):', 1),
 ('(1973).', 1),
 ('*', 3),
 ('17-21.', 1),
 ('1990', 1),
 ('2682899.', 1),
 ('27', 1),
 ('American', 1),
 ("Analysis'.", 1),
 ('Anscombe,', 1),
 ('California', 1),
 ('Census;', 1),
 ('F.', 1),
 ('J.', 1),
 ('JSTOR', 1),
 ('Statistical', 1),
 ('Statistician.', 1),
 ('This', 1)]

In [106]:
# saveAsTextFile grava o RDD em disco, ele não salva em arquivo único
# e sim em uma pasta com um arquivo por partição mais a pasta _SUCCESS
# que marca que a gravação terminou sem erro.

palavraContarOrd.saveAsTextFile("contar_palavras_out_4")

In [ ]:
sc.stop()

# 3. Leitura e Definição de Estrutura de Arquivo (Schema)

In [44]:
# diferente dos tópicos 1 e 2 aqui inicializamos uma nova sessão com SparkSession
# que é uma interface mais moderda que executa por baixo o SparkContext e adiciona
# suporte a DataFrames e Spark SQL

spark = SparkSession.builder.getOrCreate()

df = spark.read.csv('ecommerce_estatistica.csv')
df.show(5)

+--------------------+--------------------+------+--------------------+--------+------+------------------+------------------+------------------+--------------------+--------------------+--------------------+------------+-------+------------------+-------------------+------------------+------------------+---------+------------+-------------+----------------+------------------+-----------------+
|                 _c0|                 _c1|   _c2|                 _c3|     _c4|   _c5|               _c6|               _c7|               _c8|                 _c9|                _c10|                _c11|        _c12|   _c13|              _c14|               _c15|              _c16|              _c17|     _c18|        _c19|         _c20|            _c21|              _c22|             _c23|
+--------------------+--------------------+------+--------------------+--------+------+------------------+------------------+------------------+--------------------+--------------------+--------------------

In [46]:
# o spark.read.csv ler a primeira linha como dado comum, e define todos os dados
# como tipo string
df.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)
 |-- _c18: string (nullable = true)
 |-- _c19: string (nullable = true)
 |-- _c20: string (nullable = true)
 |-- _c21: string (nullable = true)
 |-- _c22: string (nullable = true)
 |-- _c23: string (nullable = true)



In [49]:
df = spark.read.option('header', 'true').csv('ecommerce_estatistica.csv')
df.show(10)

+--------------------+--------------------+--------------------+--------------------+------------------+------------------+------------------+-------------------+------------------+--------------------+--------------------+--------------------+------------------+-----------------+------------------+-------------------+------------------+------------------+---------+------------+-------------+----------------+------------------+-----------------+
|          Unnamed: 0|              Título|                Nota|        N_Avaliações|          Desconto|             Marca|          Material|             Gênero|         Temporada|             Review1|             Review2|             Review3|      Qtd_Vendidos|            Preço|       Nota_MinMax|N_Avaliações_MinMax|   Desconto_MinMax|      Preço_MinMax|Marca_Cod|Material_Cod|Temporada_Cod|Qtd_Vendidos_Cod|        Marca_Freq|    Material_Freq|
+--------------------+--------------------+--------------------+--------------------+---------------

In [52]:
# inferSchema infere, com base em uma amostra, os tipos de dados de cada coluna, para não tratar
# tudo como texto.
df = spark.read.option('header', 'true').option('inferSchema', 'true').csv('ecommerce_estatistica.csv')
df.printSchema()

root
 |-- Unnamed: 0: string (nullable = true)
 |-- Título: string (nullable = true)
 |-- Nota: string (nullable = true)
 |-- N_Avaliações: string (nullable = true)
 |-- Desconto: string (nullable = true)
 |-- Marca: string (nullable = true)
 |-- Material: string (nullable = true)
 |-- Gênero: string (nullable = true)
 |-- Temporada: string (nullable = true)
 |-- Review1: string (nullable = true)
 |-- Review2: string (nullable = true)
 |-- Review3: string (nullable = true)
 |-- Qtd_Vendidos: string (nullable = true)
 |-- Preço: string (nullable = true)
 |-- Nota_MinMax: string (nullable = true)
 |-- N_Avaliações_MinMax: double (nullable = true)
 |-- Desconto_MinMax: double (nullable = true)
 |-- Preço_MinMax: double (nullable = true)
 |-- Marca_Cod: double (nullable = true)
 |-- Material_Cod: double (nullable = true)
 |-- Temporada_Cod: double (nullable = true)
 |-- Qtd_Vendidos_Cod: double (nullable = true)
 |-- Marca_Freq: double (nullable = true)
 |-- Material_Freq: double (nullable

In [54]:
# o mode DROPMALFORMED derruba linhas mal formadas ou que não batem com o formato esperado
# dessa forma evitar quebra de dados, portanto a exibição fica mais limpa
df = spark.read.\
    option('sep', ',').\
    option('header', 'true').\
    option('mode', 'DROPMALFORMED').\
    option('inferSchema', 'true').\
    csv('ecommerce_estatistica.csv')

df.show(10)

+----------+--------------------+----+------------+--------+-------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+------------+------+------------------+-------------------+------------------+------------------+---------+------------+-------------+----------------+------------------+------------------+
|Unnamed: 0|              Título|Nota|N_Avaliações|Desconto|  Marca|            Material|    Gênero|           Temporada|             Review1|             Review2|             Review3|Qtd_Vendidos| Preço|       Nota_MinMax|N_Avaliações_MinMax|   Desconto_MinMax|      Preço_MinMax|Marca_Cod|Material_Cod|Temporada_Cod|Qtd_Vendidos_Cod|        Marca_Freq|     Material_Freq|
+----------+--------------------+----+------------+--------+-------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+------------+------+------------------+-------------------+---------

In [69]:
# o InferSchema infere os dados numéricos como double, portanto estou criando meu próprio
# schema para ter esses dados como Float
lista_campos = [
    StructField('Preço', FloatType()),
    StructField('Nota_MinMax', FloatType()),
    StructField('Desconto_MinMax', FloatType()),
    StructField('N_Avaliações_MinMax', FloatType()),
    StructField('Preço_MinMax', FloatType())
]

schema_definido = StructType(lista_campos)

df = spark.read.option('header', 'true').\
schema(schema_definido).\
csv('ecommerce_estatistica.csv')
df.show(10)
print()
df.printSchema()

+-----+-----------+---------------+-------------------+------------+
|Preço|Nota_MinMax|Desconto_MinMax|N_Avaliações_MinMax|Preço_MinMax|
+-----+-----------+---------------+-------------------+------------+
|  1.0|       NULL|            4.5|             3034.0|        18.0|
| NULL|       NULL|           NULL|               NULL|        NULL|
|  2.0|       NULL|            4.7|             5682.0|        20.0|
|  3.0|       NULL|            4.6|             1700.0|        22.0|
| NULL|       NULL|           NULL|             229.39|   0.8518519|
|  4.0|       NULL|            4.4|              507.0|         9.0|
| NULL|       NULL|           NULL|               NULL|      1000.0|
|  5.0|       NULL|            4.7|               58.0|         5.0|
| NULL|       NULL|           NULL|               NULL|        NULL|
| NULL|       NULL|           NULL|               NULL|        NULL|
+-----+-----------+---------------+-------------------+------------+
only showing top 10 rows

root
 |-

In [72]:
# Criando manualmente uma estrutura como se fosse uma linha de tabela,
# útio para criar dados de teste sem depender de um arquivo externo.

dados_teste = [
    Row(nome='Sr. Vitor Gabriel Silva', cpf='640.***.***-01', idade=90.0, data=date(1933, 12, 6),
        endereco='trevo emanuel silva, 482', bairro='vila tirol', estado='AP'),
    Row(nome='Calebe Santos', cpf='651.***.***-46', idade=40.0, data=date(1984, 3, 6),
        endereco='recanto moraes, 7', bairro='ernesto nascimento', estado='PB'),
    # ...
]

df_parcial = spark.createDataFrame(data=dados_teste, schema=('inferSchema', 'true'))
df_parcial.printSchema()


root
 |-- inferSchema: string (nullable = true)
 |-- true: string (nullable = true)
 |-- idade: double (nullable = true)
 |-- data: date (nullable = true)
 |-- endereco: string (nullable = true)
 |-- bairro: string (nullable = true)
 |-- estado: string (nullable = true)



In [74]:
spark.stop()

# 4. Leitura e Escrita de Dados: CSV, Parquet e Tabelas SQL

In [77]:
spark = SparkSession.builder.getOrCreate()

In [88]:
df = spark.read.csv('ecommerce_estatistica.csv')
df.write.\
mode('overwrite').\
csv('output/ecommerce_csv')

In [93]:
df = spark.read.option('header', 'true').\
option('inferSchema', 'true').\
option('mode','DROPMALFORMED').\
csv('output/ecommerce_csv')


df.show(5)

+--------------------+--------------------+------+--------------------+------------------+------------------+------------------+------------------+------------------+--------------------+--------------------+--------------------+------------------+-----------------+------------------+-------------------+------------------+------------------+---------+------------+-------------+----------------+------------------+-----------------+
|          Unnamed: 0|              Título|  Nota|        N_Avaliações|          Desconto|             Marca|          Material|            Gênero|         Temporada|             Review1|             Review2|             Review3|      Qtd_Vendidos|            Preço|       Nota_MinMax|N_Avaliações_MinMax|   Desconto_MinMax|      Preço_MinMax|Marca_Cod|Material_Cod|Temporada_Cod|Qtd_Vendidos_Cod|        Marca_Freq|    Material_Freq|
+--------------------+--------------------+------+--------------------+------------------+------------------+------------------+--

In [94]:
# parquet é um formato colunar muito mais eficiente que CSV para leitura analítica
# ideal para quando se trabalha com grandes volumes

df.write.option('header', 'true').save('output/ecommerce_parquet')

df = spark.read.option('header', 'true').parquet('output/ecommerce_parquet')
df.show()

+--------------------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------+------------+-------------+----------------+------------------+------------------+
|          Unnamed: 0|              Título|                Nota|        N_Avaliações|          Desconto|             Marca|            Material|             Gênero|           Temporada|             Review1|             Review2|             Review3|      Qtd_Vendidos|             Preço|       Nota_MinMax|N_Avaliações_MinMax|   Desconto_MinMax|      Preço_MinMax|Marca_Cod|Material_Cod|Temporada_Cod|Qtd_Vendidos_Cod|        Marca_Freq|     Material_Freq|
+--------------------+--------------------+--------------------+--------------------+---

In [95]:
spark.stop()